# Data loading and wrangling

Before any model, a dataset has to be understood and shaped. This notebook loads the California
Housing table, checks its structure and quality, derives a few features, and produces a clean
train/test split that the later notebooks reuse.

## Learning objectives

By the end of this notebook you will be able to:

- load a tabular dataset and inspect shape, types, and missing values;
- describe the target and its units;
- derive ratio features that carry more signal than raw counts;
- recognise outliers and decide whether to keep or trim them;
- make a reproducible train/test split with a fixed seed.

## Concept

**Wrangling** is the unglamorous work of turning a raw table into one a model can use. The first
steps never change: how many rows and columns, what type is each column, are there missing values,
and what do the ranges look like. A model fed nonsense is nonsense out, so these checks come
before any algorithm.

**Feature engineering** creates new columns from old ones. Raw counts such as `Population` are
often less useful than a ratio such as `rooms_per_person`, because a ratio is comparable across
neighbourhoods of different sizes. The module's `analysis.add_features` adds three such ratios.

**Outliers** are values far from the rest. They may be errors or they may be the most interesting
rows. Trimming them changes the question, so we inspect first and document the decision. Crucially,
any trimming or scaling must be learned on the training set only, or the test score leaks
information.

A **train/test split** holds back part of the data so the model is judged on rows it has never
seen. Fixing the random seed makes the split reproducible.

## Worked example

### Load and inspect

`load_california` reads `data/raw/california_housing.csv` (fetch it with
`python scripts/download_data.py --module 07`).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import analysis
from ds_practice import load_california, set_seed

set_seed(42)
housing = load_california()
print("shape:", housing.shape)
display(housing.head())

shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [2]:
housing.info()
print("\nmissing values per column:")
print(housing.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB

missing values per column:
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64


### Target and ranges

`MedHouseVal` is the median house value for a census block group, measured in hundreds of thousands
of dollars. The summary below shows the target is capped at 5.0 (about $500k), which is a known
artefact of the original dataset.

In [3]:
print(housing[analysis.TARGET].describe().round(3).to_string())
print("\nranges of the base features:")
display(housing[analysis.BASE_FEATURES].describe().loc[["min", "max"]].round(2))

count    20640.000
mean         2.069
std          1.154
min          0.150
25%          1.196
50%          1.797
75%          2.647
max          5.000

ranges of the base features:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
min,0.5,1.0,0.85,0.33,3.0,0.69,32.54,-124.35
max,15.0,52.0,141.91,34.07,35682.0,1243.33,41.95,-114.31


### Feature engineering

Three ratios are added. `population_per_household` restates `AveOccup` in the same units, while
`bedrooms_per_room` and `rooms_per_person` describe crowding in a way raw counts cannot.

In [4]:
housing = analysis.add_features(housing)
new_columns = [c for c in housing.columns if c not in analysis.BASE_FEATURES + [analysis.TARGET]]
print("added:", new_columns)
display(housing[new_columns].describe().round(3))

added: ['bedrooms_per_room', 'population_per_household', 'rooms_per_person']


,bedrooms_per_room,population_per_household,rooms_per_person
count,20640.000,20640.00,20640.000
mean,0.213,499.54,1.977
std,0.058,382.33,1.146
min,0.100,1.00,0.003
25%,0.175,280.00,1.522
50%,0.203,409.00,1.938
75%,0.240,605.00,2.296
max,1.000,6082.00,55.222


### Outliers

A few block groups report implausibly many rooms. We inspect rather than silently remove; if we do
trim, we record how many rows and why.

In [5]:
extreme = housing.nlargest(5, "AveRooms")[["AveRooms", "AveOccup", "Population", "MedHouseVal"]]
display(extreme)

threshold = housing["AveRooms"].quantile(0.99)
trimmed = housing[housing["AveRooms"] <= threshold]
print(f"rows above the 99th percentile of AveRooms: {len(housing) - len(trimmed)}")

,AveRooms,AveOccup,Population,MedHouseVal
1914,141.909091,2.727273,30.0,5.00001
1979,132.533333,2.400000,36.0,1.62500
12447,62.422222,1.844444,83.0,0.87500
1913,61.812500,2.333333,112.0,4.37500
11862,59.875000,1.750000,28.0,0.67500


rows above the 99th percentile of AveRooms: 207


### A reproducible split

We keep all rows (outliers included) for teaching and split 80/20 with a fixed seed. The test set
is set aside now and not touched until evaluation.

In [6]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(housing, test_size=0.2, random_state=42)
print("train:", train.shape, "| test:", test.shape)
print("mean target train/test:",
      round(train[analysis.TARGET].mean(), 3),
      round(test[analysis.TARGET].mean(), 3))

train: (16512, 12) | test: (4128, 12)
mean target train/test: 2.072 2.055


## Exercises

1. **Range check.** Report how many block groups have `AveRooms` above 20 and what share of the
   data that is. Decide whether to drop them and justify the choice in two sentences.
2. **A new ratio.** Add `bedrooms_per_person = AveBedrms / AveOccup` to a copy of the frame and
   compare its correlation with the target to that of `AveBedrms`.
3. **Split sizes.** Re-run `train_test_split` with `test_size=0.3` and `random_state=7`. Report the
   train and test row counts and explain why fixing the seed matters.

## Limitations

The target is censored at 5.0, so any model under-predicts the most expensive areas. The features
are block-group aggregates, not individual homes, which is an ecological fallacy if read as
household facts. Coordinates encode geography but also stand in for unmeasured factors, so a model
can learn location-specific patterns that will not transfer to another region.